# Solution - Exercise 2 - Give TravelMind Hands

Worked solutions in the same shape as Exercise 1:

- the idea in plain words
- a picture of what is moving
- the working code
- why it works
- the wrong turn people take

The jump from Exercise 1: you stop scripting the steps. The model decides them. So you trace first, then trust.

Run this once:

```python
```

In [ ]:
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from langchain_core.tools import tool
from langgraph.checkpoint.memory import InMemorySaver

MODEL_ID = "us.anthropic.claude-haiku-4-5-20251001-v1:0"
model = init_chat_model(MODEL_ID, model_provider="bedrock_converse", region_name="us-east-1")

BOOKINGS = {
    "JX48Q2": {"status": "CANCELLED", "seat": "14C", "tier": "Gold", "segment": "BLR-DEL"},
}

## Part A - Trace the loop

**The idea in plain words.** An agent is a model on a short leash that can reach for tools. It thinks, decides if it needs a tool, uses it, reads the result, and thinks again. It stops the moment it can answer.

**The corrected loop.** The exercise diagram sent the tool result back to the decision diamond. Wrong. After a tool runs, the result goes back to the model to reason on. The fixed arrow is `T --> M`.

```mermaid
graph TD
    U["user"] --> M["model: reason"]
    M --> D{"tool needed?"}
    D -->|"yes"| T["call tool"]
    T --> M
    D -->|"no"| A["final answer"]
```

**The single-question trace.**

```mermaid
sequenceDiagram
    participant U as user
    participant A as agent model
    participant T as get_booking
    U->>A: What tier is JX48Q2?
    A->>T: get_booking JX48Q2
    T-->>A: tier Gold and more
    A-->>U: Gold tier
```

In [ ]:
@tool
def get_booking(pnr: str) -> dict:
    """Return booking details for a PNR."""
    return BOOKINGS.get(pnr, {"error": "not found"})

agent = create_agent(model, tools=[get_booking], system_prompt="You are TravelMind.")
result = agent.invoke({"messages": [{"role": "user", "content": "What tier is JX48Q2?"}]})
print(result["messages"][-1].text())

**The answers.**

- Order: reason, call the tool, observe the result, answer.
- Tool calls: one. The tier sits in a single booking record, so one lookup returns it.
- Message trace:

| # | Role | Carries |
|---|---|---|
| 1 | human | the question |
| 2 | ai | a tool call for `get_booking` |
| 3 | tool | the result dict |
| 4 | ai | the final answer |

- Wrong arrow: `T --> D` should be `T --> M`. The observation goes back to the model.

**The wrong turn.** Reading `result["messages"][0]` for the answer. The final reply is the last message, not the first. Index with `[-1]`.

## Part B - Debug and fix

### B1

```python
@tool
def refund(pnr: str) -> str:
    return f"Refund started for {pnr}"
```

- **Broken line.** The function body has no docstring.
- **Plain why.** The docstring is the tool description the model reads to decide when to use it. No docstring means a blind tool.
- **Fix.** Add one line: `"""Start a refund for a PNR."""`.

### B2

```python
agent.invoke("status of JX48Q2?")
```

- **Broken line.** This one, a bare string.
- **Plain why.** `create_agent` runs on a state object, and that state is a message list. It does not accept a loose string.
- **Fix.** `agent.invoke({"messages": [{"role": "user", "content": "status of JX48Q2?"}]})`.

### B3

```python
agent.invoke({"messages": [{"role": "user", "content": "My PNR is JX48Q2."}]})
reply = agent.invoke({"messages": [{"role": "user", "content": "What was my PNR?"}]})
```

- **What is missing.** Memory. Each `invoke` is a fresh brain.
- **Two changes.** Pass `checkpointer=InMemorySaver()` into `create_agent`, and pass the same `config={"configurable": {"thread_id": "..."}}` on both calls so they share one thread.
- **Plain why.** The checkpointer is the notebook. The `thread_id` is the page number. Same page, same memory.

## Part C - Diagram to agent

**The idea in plain words.** Give the model two tools and a goal. You do not tell it which to call or in what order. It works that out.

```mermaid
graph TD
    U["user request"] --> M["model: reason"]
    M --> D{"tool needed?"}
    D -->|"yes"| T["call get_booking or rebook"]
    T --> O["observation"]
    O --> M
    D -->|"no"| A["final answer"]
```

In [ ]:
@tool
def get_booking(pnr: str) -> dict:
    """Return booking details for a PNR."""
    return BOOKINGS.get(pnr, {"error": "not found"})

@tool
def rebook(pnr: str, flight: str) -> str:
    """Rebook a PNR onto a new flight number."""
    return f"Rebooked {pnr} onto {flight}."

agent = create_agent(
    model,
    tools=[get_booking, rebook],
    system_prompt="You are TravelMind, a concise airline support assistant.",
)

result = agent.invoke({"messages": [
    {"role": "user", "content": "JX48Q2 was cancelled, put me on AI302"}
]})
print(result["messages"][-1].text())

**The order you did not write.**

```mermaid
sequenceDiagram
    participant U as user
    participant A as agent model
    participant G as get_booking
    participant R as rebook
    U->>A: JX48Q2 cancelled, put me on AI302
    A->>G: get_booking JX48Q2
    G-->>A: status CANCELLED
    A->>R: rebook JX48Q2 AI302
    R-->>A: confirmed
    A-->>U: rebooking confirmed
```

The model often confirms the booking first, then rebooks. You gave it two tools, not a script, and it chose the sequence. That freedom is the whole point of an agent, and also the reason you trace them in Part A before trusting them.

## Part D - Convert Strands to LangChain

**The idea in plain words.** The tool is the same in both frameworks. Only two things change: how you build the model, and how you send a message.

In [ ]:
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from langchain_core.tools import tool

@tool
def get_booking(pnr: str) -> dict:
    """Return booking details for a PNR."""
    return BOOKINGS.get(pnr, {"error": "not found"})

model = init_chat_model(
    "us.anthropic.claude-haiku-4-5-20251001-v1:0",
    model_provider="bedrock_converse",
    region_name="us-east-1",
)

agent = create_agent(model, tools=[get_booking], system_prompt="You are TravelMind.")
result = agent.invoke({"messages": [
    {"role": "user", "content": "What tier is JX48Q2, and is it cancelled?"}
]})
print(result["messages"][-1].text())

**Same tool, different envelope.**

```mermaid
graph TD
    TOOL["@tool get_booking, identical in both"] --> S["Strands: agent text call"]
    TOOL --> L["LangChain: invoke a messages list"]
```

The two lines that change are the model build and the call. Strands hides the message list behind `agent("text")`. LangChain puts the message list and the run state in your hands as `{"messages": [...]}`.

**Why the boundary matters.** That exposed state is not extra ceremony. It is the socket that memory, middleware, and LangGraph plug into. Strands optimizes for a fast agent out of the box. LangChain optimizes for one you can pry open and rewire. Neither is wrong, and knowing which you want is the real skill.

## Part E - Add memory

**The idea in plain words.** A checkpointer is the agent's notebook. A `thread_id` is which page it opens. Use the same page twice and it remembers what it wrote.

```mermaid
graph LR
    T1["turn 1: my PNR is JX48Q2"] --> CP["checkpointer saves thread rao-JX48Q2"]
    CP --> T2["turn 2, same thread: replays turn 1 plus new question"]
    T2 --> R["recalls BLR-DEL cancelled"]
```

In [ ]:
agent = create_agent(
    model,
    tools=[get_booking, rebook],
    system_prompt="You are TravelMind, a concise airline support assistant.",
    checkpointer=InMemorySaver(),
)

cfg = {"configurable": {"thread_id": "rao-JX48Q2"}}

agent.invoke({"messages": [{"role": "user", "content": "My PNR is JX48Q2."}]}, config=cfg)

reply = agent.invoke(
    {"messages": [{"role": "user", "content": "Which flight was cancelled on my booking?"}]},
    config=cfg,
)
print(reply["messages"][-1].text())

Turn 2 never repeats the PNR, yet the agent answers `BLR-DEL`. The shared `thread_id` replays turn 1 into turn 2, so the model sees the earlier message and calls `get_booking` on the remembered PNR.

**Stretch: guard the risky action.** `rebook` changes a booking, so make it ask first. Middleware wraps the loop so `rebook` pauses for a human yes before it runs. The tool code stays untouched.

```python
from langchain.agents.middleware import HumanInTheLoopMiddleware

agent = create_agent(
    model,
    tools=[get_booking, rebook],
    system_prompt="You are TravelMind.",
    checkpointer=InMemorySaver(),
    middleware=[HumanInTheLoopMiddleware(...)],  # configure rebook to require approval
)
```

## Recap

```mermaid
graph LR
    A["trace: think, act, observe, answer"] --> B["debug: docstring, message shape, memory"]
    B --> C["diagram to agent: model picks the order"]
    C --> D["Strands to LangChain: same tool, exposed state"]
    D --> E["memory: same thread_id, same page"]
```

The leap from Exercise 1: you stopped scripting the steps and handed that job to the model. Tracing first, then trusting, is how you keep that power on a leash.